In [1]:
# ============================
# Google Colab: mBART Fine-tuning for Sinhala Error Correction
# (based on your uploaded notebook, cleaned + made Colab-safe)
# ============================

# ▶️ 0) (Optional) Check GPU
!nvidia-smi -L


GPU 0: NVIDIA RTX A6000 (UUID: GPU-0dc15e44-d056-488e-02c7-e9ee7278e464)


In [2]:
# ▶️ 1) Install dependencies (Colab-friendly: don't pin torch)
!pip -q install -U transformers datasets accelerate sentencepiece evaluate sacrebleu huggingface_hub
# Optional (only if you want Weights & Biases logging)
!pip -q install -U wandb

print("✓ Installed")


✓ Installed


In [ ]:
# ▶️ 2) Login (Hugging Face + optional W&B)
import os
from dotenv import load_dotenv

# ---- Hugging Face ----
# Option A (recommended): set HF_TOKEN in Colab:
#   Runtime → (🔑 Secrets) or use:
#   os.environ["HF_TOKEN"] = "hf_...."
from huggingface_hub import login

load_dotenv()
hf_token = os.environ.get("HF_TOKEN", "").strip()
if hf_token:
    login(token=hf_token)
    print("✓ Hugging Face logged in via HF_TOKEN env")
else:
    print("⚠️ No HF_TOKEN found. If you want to push_to_hub, set HF_TOKEN first.")

# ---- Weights & Biases (optional) ----
# DISABLED due to pydantic/typing_extensions version conflicts
use_wandb = False  # set False to avoid wandb dependency issues
if use_wandb:
    import wandb
    wandb_key = os.environ.get("WANDB_API_KEY", "").strip()
    if wandb_key:
        wandb.login(key=wandb_key)
        print("✓ W&B logged in via WANDB_API_KEY env")
    else:
        print("⚠️ No WANDB_API_KEY found. Set it or keep use_wandb=False.")


✓ Hugging Face logged in via HF_TOKEN env


In [4]:
CONFIG = {
    "model_name": "facebook/mbart-large-50",
    "dataset_id": "SPEAK-ASR/sinhala-spelling-correction",

    # IMPORTANT:
    # mBART-50 language codes usually look like "si_LK".
    "source_lang": "si_LK",
    "target_lang": "si_LK",

    "max_input_length": 128,
    "max_target_length": 128,

    # Colab-safe defaults (you can increase if you have A100)
    "per_device_train_batch_size": 32, # Reduced from 8
    "per_device_eval_batch_size": 16,
    "gradient_accumulation_steps": 1,  # Increased from 4, effective batch = 4*8=32
    "auto_find_batch_size": True,  # set True if you want to
    "num_epochs": 5,
    "learning_rate": 5e-5,
    "warmup_steps": 500,
    "weight_decay": 0.01,

    # Hub push (using official HF train/test splits)
    "push_to_hub": True,                 # requires HF_TOKEN
    "hub_model_id": "SPEAK-ASR/mBART-large-50-si-spelling-v2",  # NEW: separate model name

}

In [5]:
# ▶️ 3) Import core libraries
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [6]:
# ▶️ 4) Load dataset
from datasets import load_dataset
print("[STEP 1] Loading dataset:", CONFIG["dataset_id"])
dataset = load_dataset(CONFIG["dataset_id"])
print("✓ Loaded splits:", list(dataset.keys()))
print("Train samples:", len(dataset["train"]))
print("Columns:", dataset["train"].column_names)
print("Sample:", dataset["train"][0])


[STEP 1] Loading dataset: SPEAK-ASR/sinhala-spelling-correction


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/506k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/37712 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9428 [00:00<?, ? examples/s]

✓ Loaded splits: ['train', 'test']
Train samples: 37712
Columns: ['dyslexic_sentence', 'clean_sentence']
Sample: {'dyslexic_sentence': 'අද සන්දව බොහොම ලස්සනයි', 'clean_sentence': 'අද සන්ධ්\u200dයාව බොහොම ලස්සනයි'}


In [7]:
dataset['train'][0]

{'dyslexic_sentence': 'අද සන්දව බොහොම ලස්සනයි',
 'clean_sentence': 'අද සන්ධ්\u200dයාව බොහොම ලස්සනයි'}

In [8]:
# ▶️ 5) Load tokenizer + model
print("[STEP 2] Loading model/tokenizer:", CONFIG["model_name"])

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], use_fast=False)
tokenizer.src_lang = CONFIG["source_lang"]
tokenizer.tgt_lang = CONFIG["target_lang"]

model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"])

# Set language tokens for mBART generation
lang_token_id = tokenizer.convert_tokens_to_ids([CONFIG["target_lang"]])[0]
model.config.decoder_start_token_id = lang_token_id
# model.config.forced_bos_token_id = lang_token_id

model = model.to(device)

print("✓ Model ready")
print("Vocab size:", len(tokenizer))
print("decoder_start_token_id:", model.config.decoder_start_token_id)


[STEP 2] Loading model/tokenizer: facebook/mbart-large-50


Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✓ Model ready
Vocab size: 250054
decoder_start_token_id: 250022


In [9]:
# ▶️ 6) Preprocess + tokenize (using official HF train/test splits)
print("[STEP 3] Tokenizing & preparing datasets...")

# Detect dataset column names (supports both possibilities from your notebook)
# input: "input_text" or "dyslexic_sentence"
# target: "corrected_text" or "clean_sentence"
train_cols = dataset["train"].column_names
input_col = "input_text" if "input_text" in train_cols else ("dyslexic_sentence" if "dyslexic_sentence" in train_cols else None)
target_col = "corrected_text" if "corrected_text" in train_cols else ("clean_sentence" if "clean_sentence" in train_cols else None)

if input_col is None or target_col is None:
    raise ValueError(f"Could not find expected columns. Found: {train_cols}")

print("Using columns:")
print("  input_col =", input_col)
print("  target_col =", target_col)

def preprocess_function(examples):
    input_texts, target_texts = [], []

    for i in range(len(examples[input_col])):
        src = examples[input_col][i]
        tgt = examples[target_col][i]

        if src and tgt and str(src).strip() and str(tgt).strip():
            input_texts.append(str(src))
            target_texts.append(str(tgt))

    if len(input_texts) == 0:
        return {"input_ids": [], "attention_mask": [], "labels": []}

    # Encode inputs
    model_inputs = tokenizer(
        input_texts,
        max_length=CONFIG["max_input_length"],
        padding="max_length",
        truncation=True,
    )

    # ✅ NEW transformers-compatible target encoding
    labels = tokenizer(
        text_target=target_texts,
        max_length=CONFIG["max_target_length"],
        padding="max_length",
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


# Map tokenization to all splits (train and test from HF)
dataset_tok = dataset.map(
    preprocess_function,
    batched=True,
    batch_size=100,
    remove_columns=train_cols,
    desc="Tokenizing",
)

# Filter empties (extra safety)
def keep_nonempty(ex):
    return ex["input_ids"] is not None and len(ex["input_ids"]) > 0 and ex["labels"] is not None and len(ex["labels"]) > 0

dataset_tok = dataset_tok.filter(keep_nonempty)

# --- Use official HF splits: train for training, test for evaluation ---
# NOTE: We'll use a small fraction of the train split for validation during training
train_dataset = dataset_tok["train"]
test_dataset = dataset_tok["test"]

# Split train into train+eval for during-training validation
eval_ratio = 0.15  # use 15% of train for eval during training
split = train_dataset.train_test_split(test_size=eval_ratio, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print("✓ Split sizes:")
print("  Train (for training) :", len(train_dataset))
print("  Eval  (during training):", len(eval_dataset))
print("  Test  (official HF test):", len(test_dataset))
print("Columns:", train_dataset.column_names)
print("Sample tokenized row:", train_dataset[0])

[STEP 3] Tokenizing & preparing datasets...
Using columns:
  input_col = dyslexic_sentence
  target_col = clean_sentence


Tokenizing:   0%|          | 0/37712 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/9428 [00:00<?, ? examples/s]

Filter:   0%|          | 0/37712 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9428 [00:00<?, ? examples/s]

✓ Split sizes:
  Train (for training) : 32055
  Eval  (during training): 5657
  Test  (official HF test): 9428
Columns: ['input_ids', 'attention_mask', 'labels']
Sample tokenized row: {'input_ids': [250022, 60583, 74911, 110579, 6, 249540, 919, 5056, 3611, 204582, 56, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [10]:
# ▶️ 7) Metrics (optional but useful)
import numpy as np
import evaluate

sacrebleu = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [[l.strip()] for l in labels]  # sacrebleu expects list of references per pred
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels_pp = postprocess_text(decoded_preds, decoded_labels)

    bleu = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels_pp)["score"]

    # exact match
    exact = np.mean([p == l[0] for p, l in zip(decoded_preds, decoded_labels_pp)])

    return {"bleu": bleu, "exact_match": float(exact)}


In [11]:
# ▶️ 8) Training setup (FULLY CORRECTED for new + old Transformers)
from transformers import set_seed, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, EarlyStoppingCallback
import torch

set_seed(42)

output_dir = "/content/mbart-model"

# bf16 works on A100; fp16 works on T4/V100
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8  # Ampere+
use_fp16 = torch.cuda.is_available() and not use_bf16

report_to = ["wandb"] if use_wandb else []

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=CONFIG["num_epochs"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],

    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],

    # ✅ NEW API name (your Colab needs this)
    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,

    push_to_hub=CONFIG["push_to_hub"],
    hub_model_id=CONFIG["hub_model_id"],
    hub_strategy="checkpoint",


    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=CONFIG["max_target_length"],


    fp16=use_fp16,
    bf16=use_bf16,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,



    report_to=report_to,
    dataloader_num_workers=4,
    dataloader_pin_memory = True,
    remove_unused_columns=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# ✅ Trainer: new transformers uses `processing_class` instead of `tokenizer`
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

try:
    # New transformers
    trainer = Seq2SeqTrainer(**trainer_kwargs, processing_class=tokenizer)
except TypeError:
    # Older transformers fallback
    trainer = Seq2SeqTrainer(**trainer_kwargs, tokenizer=tokenizer)

print("✓ Trainer ready")
print("fp16:", use_fp16, "| bf16:", use_bf16)


✓ Trainer ready
fp16: False | bf16: True


In [12]:
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ CUDA cache cleared")

✓ CUDA cache cleared


In [13]:
model.config.forced_bos_token_id = None
model.generation_config.forced_bos_token_id = lang_token_id

In [14]:
# ▶️ 9) Train
print("[STEP 5] Training...")
# Try to clear CUDA cache before training to free up any fragmented memory

train_result = trainer.train()
print("✓ Training done")
print("Training loss:", train_result.training_loss)

# Save locally
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Saved to:", output_dir)

[STEP 5] Training...


Step,Training Loss,Validation Loss,Bleu,Exact Match
500,0.040321,0.040790,58.006304,0.407637
1000,0.032524,0.031655,63.429224,0.470744
1500,0.024066,0.029188,66.130168,0.501856
2000,0.021291,0.026869,68.422738,0.530493
2500,0.010920,0.030166,67.791750,0.524306
3000,0.011154,0.028481,68.631499,0.536680
3500,0.006546,0.030878,69.097152,0.542337


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved to: /content/mbart-model


In [15]:
# ▶️ 10) Evaluate on test set + show examples
print("[STEP 6] Testing...")

pred_out = trainer.predict(test_dataset)
decoded_preds = tokenizer.batch_decode(pred_out.predictions, skip_special_tokens=True)

labels = np.where(pred_out.label_ids != -100, pred_out.label_ids, tokenizer.pad_token_id)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

print("\nSample predictions:")
for i in range(min(5, len(decoded_preds))):
    print("\n---", i+1, "---")
    print("PRED:", decoded_preds[i])
    print("GOLD:", decoded_labels[i])
    print("MATCH:", "✓" if decoded_preds[i].strip() == decoded_labels[i].strip() else "✗")

# quick exact-match
exact = np.mean([p.strip() == l.strip() for p, l in zip(decoded_preds, decoded_labels)])
print("\nExact match (test):", round(float(exact)*100, 2), "%")


[STEP 6] Testing...



Sample predictions:

--- 1 ---
PRED: පොත් පොතක කතා ලියලා තියෙනවා
GOLD: පොත් පොතක කතා ලියලා තියෙනවා
MATCH: ✓

--- 2 ---
PRED: මට බඩගින්නේ නෑ
GOLD: මට බඩගිනි නෑ
MATCH: ✗

--- 3 ---
PRED: විදේශ සංචිතය පසුගිය මාස
GOLD: විදේශ සංචිතය පසුගිය මාර්තු
MATCH: ✗

--- 4 ---
PRED: අම්මා කුස්සියේ උයනවා
GOLD: අම්මා කුස්සියේ උයනවා
MATCH: ✓

--- 5 ---
PRED: සොයිර ගහ ලස්සනයි
GOLD: සොයිර ගහ ලස්සනයි
MATCH: ✓

Exact match (test): 52.91 %


In [16]:
# ▶️ 11) Push to Hugging Face Hub (if logged in)
if CONFIG["push_to_hub"] and os.environ.get("HF_TOKEN",""):
    print("Pushing to hub:", CONFIG["hub_model_id"])
    trainer.push_to_hub(
        dataset=CONFIG["dataset_id"],
        language="si",
        finetuned_from=CONFIG["model_name"],
        model_name=CONFIG["hub_model_id"],
    )
    print("✓ Pushed")
else:
    print("Skipping push_to_hub (no HF_TOKEN or push_to_hub=False).")


Skipping push_to_hub (no HF_TOKEN or push_to_hub=False).
